# Exploratory Data Analysis (EDA)
## Professional League of Legends Player Career Trajectories

This notebook conducts EDA on player career data to understand patterns, distributions, and relationships in professional esports careers.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Color palette (League of Legends inspired - pastel blue & gold)
COLORS = {
    'blue': '#5b8bd4',
    'blue_dark': '#3d6cb3',
    'blue_light': '#a8c5e8',
    'gold': '#d4a74a',
    'gold_dark': '#c9a227',
    'gold_light': '#e8d4a0'
}

print("Libraries loaded successfully")

---
## 1. Dataset Summary

Load and summarize the dataset including target variable, features, and data types.

In [ ]:
# Try to load real data, otherwise generate sample data
DATA_PATH = Path('../data/processed')

def load_or_generate_data():
    """Load processed data or generate sample data for EDA demonstration."""
    
    players_file = DATA_PATH / 'players_processed.csv'
    careers_file = DATA_PATH / 'career_metrics.csv'
    
    if players_file.exists() and careers_file.exists():
        print("Loading real data...")
        players = pd.read_csv(players_file)
        careers = pd.read_csv(careers_file)
        is_sample = False
    else:
        print("Generating sample data for EDA demonstration...")
        players, careers = generate_sample_data(n=500)
        is_sample = True
    
    return players, careers, is_sample

def generate_sample_data(n=500):
    """Generate realistic sample data based on expected distributions."""
    np.random.seed(42)
    
    regions = ['LCK', 'LPL', 'LEC', 'LCS', 'PCS', 'VCS', 'CBLOL', 'LJL', 'LLA']
    region_weights = [0.18, 0.22, 0.15, 0.12, 0.08, 0.09, 0.07, 0.05, 0.04]
    roles = ['Top', 'Jungle', 'Mid', 'ADC', 'Support']
    
    # Regional career length parameters (mean months)
    region_means = {'LCK': 32, 'LPL': 30, 'LEC': 26, 'LCS': 24, 'PCS': 22, 
                    'VCS': 20, 'CBLOL': 18, 'LJL': 16, 'LLA': 15}
    
    primary_regions = np.random.choice(regions, n, p=region_weights)
    
    # Generate career lengths with regional variation
    career_months = []
    for region in primary_regions:
        mean = region_means[region]
        length = np.random.exponential(scale=mean * 0.8) + 6
        career_months.append(min(max(length, 3), 120))
    
    career_months = np.array(career_months)
    starting_tiers = np.random.choice([1, 2], n, p=[0.35, 0.65])
    
    # Promotion logic
    promoted = []
    time_to_tier1 = []
    for i in range(n):
        if starting_tiers[i] == 1:
            promoted.append(False)
            time_to_tier1.append(0)
        else:
            prob = min(0.45, career_months[i] / 70)
            is_promoted = np.random.random() < prob
            promoted.append(is_promoted)
            time_to_tier1.append(np.random.uniform(6, min(36, career_months[i] * 0.5)) if is_promoted else np.nan)
    
    players = pd.DataFrame({
        'player_id': [f'player_{i:04d}' for i in range(n)],
        'primary_role': np.random.choice(roles, n),
        'nationality': np.random.choice(['KR', 'CN', 'US', 'DE', 'DK', 'BR', 'VN', 'JP', 'TW'], n),
        'current_status': np.random.choice(['Active', 'Retired', 'Inactive'], n, p=[0.25, 0.55, 0.20]),
        'peak_tier': np.where(np.array(promoted) | (starting_tiers == 1), 1, 2)
    })
    
    careers = pd.DataFrame({
        'player_id': players['player_id'],
        'primary_region': primary_regions,
        'career_length_months': career_months,
        'career_length_years': career_months / 12,
        'time_to_tier1_months': time_to_tier1,
        'num_teams': np.random.poisson(lam=2.5, size=n) + 1,
        'num_regions': np.random.choice([1, 2, 3], n, p=[0.72, 0.23, 0.05]),
        'promoted_tier2_to_tier1': promoted,
        'starting_tier': starting_tiers,
        'career_start_year': np.random.choice(range(2013, 2024), n)
    })
    
    return players, careers

# Load data
players, careers, is_sample = load_or_generate_data()

# Merge for complete analysis
df = careers.merge(players, on='player_id')

print(f"\nDataset: {'SAMPLE DATA' if is_sample else 'REAL DATA'}")
print(f"Total Players: {len(df)}")

In [ ]:
# Dataset Summary
print("="*60)
print("DATASET SUMMARY")
print("="*60)

print("\n** Target Variable **")
print("- career_length_years: Duration of professional career (continuous)")

print("\n** Features **")
print(df.dtypes.to_string())

print("\n** Shape **")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n** First 5 Rows **")
df.head()

In [ ]:
# Detailed data types and non-null counts
print("\n** Data Types & Non-Null Counts **")
df.info()

---
## 2. Data Quality: Missing, Duplicate, and Inconsistent Data

In [ ]:
print("="*60)
print("DATA QUALITY ASSESSMENT")
print("="*60)

# Missing values
print("\n** Missing Values **")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
print(missing_df[missing_df['Missing'] > 0].to_string() if missing.sum() > 0 else "No missing values in core columns")

# Note on time_to_tier1_months
print(f"\nNote: time_to_tier1_months has {missing_df.loc['time_to_tier1_months', 'Missing']} NaN values")
print("      (Expected: NaN for players who never reached Tier 1)")

In [ ]:
# Duplicate check
print("\n** Duplicate Records **")
duplicates = df.duplicated(subset=['player_id']).sum()
print(f"Duplicate player_ids: {duplicates}")

if duplicates > 0:
    print("Removing duplicates...")
    df = df.drop_duplicates(subset=['player_id'], keep='first')
    print(f"New shape: {df.shape}")

In [ ]:
# Data consistency checks
print("\n** Data Consistency Checks **")

# Check career length validity
invalid_career = (df['career_length_months'] <= 0).sum()
print(f"Invalid career lengths (<=0): {invalid_career}")

# Check tier consistency
tier_mismatch = ((df['starting_tier'] == 1) & (df['promoted_tier2_to_tier1'] == True)).sum()
print(f"Tier 1 starters marked as promoted (inconsistent): {tier_mismatch}")

# Check categorical values
print(f"\nUnique Regions: {df['primary_region'].nunique()} - {sorted(df['primary_region'].unique())}")
print(f"Unique Roles: {df['primary_role'].nunique()} - {sorted(df['primary_role'].unique())}")
print(f"Unique Statuses: {df['current_status'].nunique()} - {sorted(df['current_status'].unique())}")

---
## 3. Descriptive Statistics

In [ ]:
print("="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)

# Numerical columns
numerical_cols = ['career_length_months', 'career_length_years', 'time_to_tier1_months', 'num_teams', 'num_regions']

print("\n** Numerical Variables Summary **")
df[numerical_cols].describe().round(2)

In [ ]:
# Key metrics for target variable
print("\n** Career Length Statistics (Target Variable) **")
career_stats = {
    'Mean': df['career_length_years'].mean(),
    'Median': df['career_length_years'].median(),
    'Std Dev': df['career_length_years'].std(),
    'Min': df['career_length_years'].min(),
    'Max': df['career_length_years'].max(),
    'IQR': df['career_length_years'].quantile(0.75) - df['career_length_years'].quantile(0.25),
    'Skewness': df['career_length_years'].skew()
}

for stat, value in career_stats.items():
    print(f"{stat}: {value:.3f}")

In [ ]:
# Career length by region
print("\n** Career Length by Region **")
region_stats = df.groupby('primary_region')['career_length_years'].agg(['mean', 'median', 'std', 'count'])
region_stats = region_stats.sort_values('mean', ascending=False).round(2)
region_stats.columns = ['Mean', 'Median', 'Std', 'Count']
print(region_stats.to_string())

In [ ]:
# Categorical variable distributions
print("\n** Categorical Variable Distributions **")

print("\nRegion Distribution:")
print(df['primary_region'].value_counts().to_string())

print("\nRole Distribution:")
print(df['primary_role'].value_counts().to_string())

print("\nCareer Status Distribution:")
print(df['current_status'].value_counts().to_string())

print("\nStarting Tier Distribution:")
print(df['starting_tier'].value_counts().to_string())

In [ ]:
# Correlation analysis
print("\n** Correlation Matrix (Numerical Variables) **")
corr_cols = ['career_length_months', 'num_teams', 'num_regions', 'starting_tier', 'career_start_year']
correlation_matrix = df[corr_cols].corr().round(3)
print(correlation_matrix.to_string())

---
## 4. Visualizations

In [ ]:
# Figure 1: Career Length Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['career_length_years'], bins=25, color=COLORS['blue'], edgecolor='white', alpha=0.8)
axes[0].axvline(df['career_length_years'].mean(), color=COLORS['gold_dark'], linestyle='--', linewidth=2, label=f"Mean: {df['career_length_years'].mean():.2f}")
axes[0].axvline(df['career_length_years'].median(), color=COLORS['gold'], linestyle=':', linewidth=2, label=f"Median: {df['career_length_years'].median():.2f}")
axes[0].set_xlabel('Career Length (Years)')
axes[0].set_ylabel('Number of Players')
axes[0].set_title('Distribution of Career Lengths')
axes[0].legend()

# Box plot by region
region_order = df.groupby('primary_region')['career_length_years'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='primary_region', y='career_length_years', order=region_order,
            palette=[COLORS['gold'] if i < 2 else COLORS['blue'] for i in range(len(region_order))], ax=axes[1])
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Career Length (Years)')
axes[1].set_title('Career Length Distribution by Region')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/eda_fig1_career_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 1: Career length shows right-skewed distribution (skewness: {:.2f})".format(df['career_length_years'].skew()))
print("         LCK and LPL show longer median careers than other regions.")

In [ ]:
# Figure 2: Tier Transition Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart - promotion outcomes for Tier 2 starters
tier2_df = df[df['starting_tier'] == 2]
promotion_counts = tier2_df['promoted_tier2_to_tier1'].value_counts()
labels = ['Did Not Promote', 'Promoted to Tier 1']
colors_pie = [COLORS['blue_light'], COLORS['gold']]
axes[0].pie(promotion_counts, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90,
            explode=(0, 0.05), shadow=False)
axes[0].set_title(f'Tier 2 Player Outcomes (n={len(tier2_df)})')

# Bar chart - promotion rate by region
promo_by_region = tier2_df.groupby('primary_region')['promoted_tier2_to_tier1'].mean() * 100
promo_by_region = promo_by_region.sort_values(ascending=False)
colors_bar = [COLORS['gold'] if v > 25 else COLORS['blue'] for v in promo_by_region]
promo_by_region.plot(kind='bar', ax=axes[1], color=colors_bar, edgecolor='white')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Promotion Rate (%)')
axes[1].set_title('Tier 2 to Tier 1 Promotion Rate by Region')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(y=25, color=COLORS['gold_dark'], linestyle='--', alpha=0.7, label='25% threshold')

plt.tight_layout()
plt.savefig('../reports/eda_fig2_tier_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

promo_rate = tier2_df['promoted_tier2_to_tier1'].mean() * 100
print(f"Figure 2: Overall Tier 2 to Tier 1 promotion rate: {promo_rate:.1f}%")
print(f"         Top regions: {promo_by_region.head(3).to_dict()}")

In [ ]:
# Figure 3: Correlation Heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Create correlation matrix with more variables
df['is_tier1'] = (df['peak_tier'] == 1).astype(int)
df['is_retired'] = (df['current_status'] == 'Retired').astype(int)

corr_vars = ['career_length_months', 'num_teams', 'num_regions', 'starting_tier', 
             'career_start_year', 'is_tier1', 'is_retired']
corr_matrix = df[corr_vars].corr()

# Custom colormap (blue to gold)
cmap = sns.diverging_palette(220, 45, as_cmap=True)

sns.heatmap(corr_matrix, annot=True, cmap=cmap, center=0, fmt='.2f',
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Correlation Heatmap of Career Variables')

plt.tight_layout()
plt.savefig('../reports/eda_fig3_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Key correlations
print("Figure 3: Key Correlations with Career Length:")
career_corr = corr_matrix['career_length_months'].drop('career_length_months').sort_values(key=abs, ascending=False)
for var, corr in career_corr.items():
    print(f"         {var}: {corr:.3f}")

In [ ]:
# Figure 4: Career Length by Role and Region Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by role
role_order = df.groupby('primary_role')['career_length_years'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='primary_role', y='career_length_years', order=role_order,
            palette=[COLORS['blue'], COLORS['gold'], COLORS['blue'], COLORS['gold'], COLORS['blue']], ax=axes[0])
axes[0].set_xlabel('Role')
axes[0].set_ylabel('Career Length (Years)')
axes[0].set_title('Career Length Distribution by Role')

# Heatmap: Mean career length by region and role
pivot_table = df.pivot_table(values='career_length_years', index='primary_region', 
                             columns='primary_role', aggfunc='mean')
pivot_table = pivot_table.loc[region_order]  # Sort by overall median
sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1],
            cbar_kws={'label': 'Mean Career (Years)'})
axes[1].set_title('Mean Career Length: Region x Role')

plt.tight_layout()
plt.savefig('../reports/eda_fig4_role_region_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 4: Role differences are less pronounced than regional differences.")
print(f"         Longest careers by role: {df.groupby('primary_role')['career_length_years'].mean().idxmax()}")

In [ ]:
# Figure 5: Career Trends Over Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Players entering per year
entry_counts = df['career_start_year'].value_counts().sort_index()
entry_counts.plot(kind='bar', ax=axes[0], color=COLORS['blue'], edgecolor='white')
axes[0].set_xlabel('Career Start Year')
axes[0].set_ylabel('Number of New Players')
axes[0].set_title('New Professional Players per Year')
axes[0].tick_params(axis='x', rotation=45)

# Average career length by start year cohort (only for retired players)
retired_df = df[df['current_status'] == 'Retired']
cohort_career = retired_df.groupby('career_start_year')['career_length_years'].mean()
cohort_career.plot(kind='line', marker='o', ax=axes[1], color=COLORS['gold_dark'], linewidth=2, markersize=8)
axes[1].fill_between(cohort_career.index, cohort_career.values, alpha=0.3, color=COLORS['gold_light'])
axes[1].set_xlabel('Career Start Year')
axes[1].set_ylabel('Mean Career Length (Years)')
axes[1].set_title('Mean Career Length by Start Year (Retired Players)')
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../reports/eda_fig5_temporal_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 5: Note - Recent cohorts show shorter careers partly due to censoring (still active).")

---
## 5. Preliminary Insights & Hypothesis Formation

In [ ]:
print("="*60)
print("PRELIMINARY INSIGHTS & HYPOTHESES")
print("="*60)

print("""
** Key Observations from EDA **

1. CAREER LENGTH DISTRIBUTION
   - Right-skewed distribution with median ({:.2f} yrs) < mean ({:.2f} yrs)
   - Most careers concentrated in 1-3 year range
   - Long careers (5+ years) are exceptional outliers
   -> Hypothesis: Esports careers follow exponential decay pattern

2. REGIONAL DIFFERENCES  
   - LCK and LPL show longest average careers
   - Smaller regions (LLA, LJL) have shortest careers
   - Gap between top and bottom regions: ~{:.1f} years
   -> Hypothesis: Established infrastructure correlates with longer careers

3. TIER TRANSITIONS
   - Only {:.1f}% of Tier 2 players reach Tier 1
   - Promotion rates vary significantly by region
   - LCK/LPL have highest promotion rates
   -> Hypothesis: Developmental system quality affects promotion odds

4. ROLE IMPACT
   - Minimal career length differences between roles
   - Regional effects dominate over role effects
   -> Hypothesis: Position is not a major longevity factor

5. CORRELATIONS
   - Number of teams correlates positively with career length (more teams = longer career)
   - Starting tier shows expected relationship with outcomes
   -> Hypothesis: Team mobility may indicate career sustainability

** Questions for Further Analysis **
- Does early promotion predict longer overall career?
- Are career lengths trending up or down over time?
- What factors predict successful Tier 2 to Tier 1 transitions?
""".format(
    df['career_length_years'].median(),
    df['career_length_years'].mean(),
    region_stats['Mean'].max() - region_stats['Mean'].min(),
    tier2_df['promoted_tier2_to_tier1'].mean() * 100
))

In [ ]:
# Summary statistics for PDF report
print("\n" + "="*60)
print("SUMMARY STATISTICS FOR REPORT")
print("="*60)

summary_stats = {
    'Total Players': len(df),
    'Mean Career (Years)': f"{df['career_length_years'].mean():.2f}",
    'Median Career (Years)': f"{df['career_length_years'].median():.2f}",
    'Std Dev (Years)': f"{df['career_length_years'].std():.2f}",
    'Tier 2 to Tier 1 Rate': f"{tier2_df['promoted_tier2_to_tier1'].mean()*100:.1f}%",
    'Top Region (Career)': region_stats['Mean'].idxmax(),
    'Longest Region Mean': f"{region_stats['Mean'].max():.2f} yrs",
    'Shortest Region Mean': f"{region_stats['Mean'].min():.2f} yrs",
    'Active Players': f"{(df['current_status']=='Active').sum()} ({(df['current_status']=='Active').mean()*100:.1f}%)",
    'Retired Players': f"{(df['current_status']=='Retired').sum()} ({(df['current_status']=='Retired').mean()*100:.1f}%)"
}

for key, value in summary_stats.items():
    print(f"{key}: {value}")

In [ ]:
# Create reports directory if needed
import os
os.makedirs('../reports', exist_ok=True)

print("\nFigures saved to ../reports/ directory:")
print("- eda_fig1_career_distribution.png")
print("- eda_fig2_tier_transitions.png")
print("- eda_fig3_correlation_heatmap.png")
print("- eda_fig4_role_region_analysis.png")
print("- eda_fig5_temporal_trends.png")